# Fine-tuning PubMedBERT for Negation and Speculation Detection

This notebook fine-tunes a PubMedBERT model to detect negation and speculation scopes in biomedical text. The model is trained on the BioScope corpus and augmented with the SFU Review Corpus, then evaluated on biomedical articles from PubMed Central.

## Import Libraries and Extract Data

In [2]:
# Import libraries and extract data
import zipfile

with zipfile.ZipFile("/content/bioscope.zip", "r") as z:
    z.extractall("/content/bioscope_extracted")
    print(z.namelist())

import copy
import glob
import io
import random
import re
import xml.etree.ElementTree as ET

import requests
import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

IGNORE_INDEX = -100
print("Imports OK. CUDA available:", torch.cuda.is_available())

['Annotation guidelines2.1.pdf', 'bioscope_cameraready.pdf', 'full_papers.xml', 'BioScope.dtd', 'abstracts.xml', 'clinical_merger/', 'clinical_merger/lib/', 'clinical_merger/lib/sax2.jar', 'clinical_merger/clinical_records_anon.xml', 'clinical_merger/ScopeMerger.jar', 'clinical_merger/BioScope.dtd', 'clinical_merger/merger.bat', 'clinical_merger/def_name_merger.bat', 'clinical_merger/README.txt']
Imports OK. CUDA available: True


## Set Training Configuration

In [ ]:
# Set training configuration
bioscope_xml  = "/content/bioscope_extracted/abstracts.xml"  
backbone      = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
epochs        = 15
batch_size    = 8
lr            = 2e-5
weight_decay  = 0.05

warmup_ratio  = 0.1
freeze_layers = 6
dropout       = 0.15
label_smoothing = 0.0
max_len       = 128
patience      = 3
val_frac      = 0.15
test_frac     = 0.15
seed          = 42
out_path      = "best_model.pt"

random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"backbone={backbone}  epochs={epochs}  batch={batch_size}  lr={lr}  device={device}")


backbone=microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext  epochs=15  batch=8  lr=2e-05  device=cuda


## Define Parsing and Preprocessing Helpers

In [ ]:
# Define parsing and preprocessing helpers
def _find_task(xcope_elem):
    """An <xcope> can contain a <cue type="negation"|"speculation">
    anywhere inside it; that cue's type tells us which scope this is."""
    for cue in xcope_elem.iter():
        if cue.tag.lower() == "cue":
            t = cue.get("type", "").strip().lower()
            if t:
                return t
    return "negation"


def _parse_bioscope_stream(raw_text):
    """
    Parse a BioScope-style XML string (cue/xcope tag scheme) into a list of
    sentence dicts: {tokens, negation_scope, speculation_scope}.
    """
    root = ET.fromstring(raw_text)
    examples = []

    def walk(elem, in_neg, in_spec, tokens, neg_scope, spec_scope):
        if elem.text:
            for w in elem.text.split():
                tokens.append(w)
                neg_scope.append(1 if in_neg else 0)
                spec_scope.append(1 if in_spec else 0)
        for child in elem:
            tag = child.tag.lower()
            child_neg, child_spec = in_neg, in_spec
            if tag == "xcope":
                task = _find_task(child)
                if task == "negation":
                    child_neg = True
                elif task == "speculation":
                    child_spec = True
            walk(child, child_neg, child_spec, tokens, neg_scope, spec_scope)
            if child.tail:
                for w in child.tail.split():
                    tokens.append(w)
                    neg_scope.append(1 if in_neg else 0)
                    spec_scope.append(1 if in_spec else 0)

    for sent in root.iter():
        if sent.tag.lower() != "sentence":
            continue
        tokens, neg_scope, spec_scope = [], [], []
        walk(sent, False, False, tokens, neg_scope, spec_scope)
        if tokens:
            examples.append({
                "tokens": tokens,
                "negation_scope": neg_scope,
                "speculation_scope": spec_scope,
            })

    return examples


def _has_sentence_tags(raw_text):
    """
    Cheap pre-check used in Cell 6: does this file contain any <sentence>
    tags at all? Some SFU files apparently have none, in which case we fall
    back to _split_into_pseudo_sentences instead of the tag-based parser.
    """
    return re.search(r"<\s*sentence", raw_text, flags=re.IGNORECASE) is not None


_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")


def _split_into_pseudo_sentences(raw_text):
    """
    Fallback splitter for files with no sentence tags at all. Splits on
    punctuation the same way the later PMC-inference code does, so each
    chunk is short enough to avoid being silently truncated by max_len, and
    small enough that wrapping it individually in <sentence> tags gives the
    parser something to key off of.
    """
    return [s.strip() for s in _SENTENCE_SPLIT_RE.split(raw_text) if s.strip()]


def _preview_example(ex):
    """Annotate a sentence with [NEG] / [SPEC] / [NEG+SPEC] markers."""
    marked = []
    for word, n, s in zip(ex["tokens"], ex["negation_scope"], ex["speculation_scope"]):
        if n and s:
            marked.append(f"[NEG+SPEC]{word}")
        elif n:
            marked.append(f"[NEG]{word}")
        elif s:
            marked.append(f"[SPEC]{word}")
        else:
            marked.append(word)
    return " ".join(marked)

## Parse the BioScope Dataset

In [5]:
# Parse the BioScope dataset
print(f"Parsing {bioscope_xml} ...")
with open(bioscope_xml, encoding="utf-8") as f:
    raw = f.read()

examples = _parse_bioscope_stream(raw)
print(f"Parsed {len(examples)} sentences.")
if examples:
    print("First example tokens:", examples[0]["tokens"][:15])

Parsing /content/bioscope_extracted/abstracts.xml ...
Parsed 11871 sentences.
First example tokens: ['Induction', 'of', 'NF-KB', 'during', 'monocyte', 'differentiation', 'by', 'HIV', 'type', '1', 'infection.']


## Split Data into Train, Validation, and Test Sets

In [6]:
# Split data into train, validation, and test sets
random.shuffle(examples)

n      = len(examples)
n_test = int(n * test_frac)
n_val  = int(n * val_frac)
test_ex  = examples[:n_test]
val_ex   = examples[n_test:n_test + n_val]
train_ex = examples[n_test + n_val:]

print(f"Train={len(train_ex)}  Val={len(val_ex)}  Test={len(test_ex)}")

for split_name, split_examples in [("TRAIN", train_ex), ("VAL", val_ex), ("TEST", test_ex)]:
    print(f"\n--- {split_name} sample sentences ---")
    for ex in split_examples[:3]:
        print(" ", _preview_example(ex))

# Leakage check
train_sigs = {" ".join(ex["tokens"]) for ex in train_ex}
val_sigs   = {" ".join(ex["tokens"]) for ex in val_ex}
test_sigs  = {" ".join(ex["tokens"]) for ex in test_ex}

print("\n--- Split leakage check ---")
for a_name, a, b_name, b in [
    ("train", train_sigs, "test",  test_sigs),
    ("train", train_sigs, "val",   val_sigs),
    ("val",   val_sigs,   "test",  test_sigs),
]:
    overlap = a & b
    print(f"  {a_name} ∩ {b_name} = {len(overlap)} sentences "
          f"({len(overlap) / max(len(b), 1):.1%} of {b_name})")

Train=8311  Val=1780  Test=1780

--- TRAIN sample sentences ---
  TRAF2 is a signal transducing adaptor molecule which binds to the CD40 cytoplasmic domain.
  Tumor necrosis factor alpha (TNF-alpha) mRNA production was analyzed by polymerase chain reaction amplification in monocytic U937 cells and in a chronically HIV infected U937 cell line (U9-IIIB).
  Now we have examined the effect of the nature of the monocyte activating signal on these two inhibitory mechanims.

--- VAL sample sentences ---
  Furthermore, a mutant E6 motif lost both in vitro binding activity and in vivo enhancer activity.
  NF-kappaB activation is required for C5a-induced interleukin-8 gene expression in mononuclear cells.
  In addition, we demonstrate that the inducible transcriptional activity of the FasL reporter is abolished by specific inhibitors of NF-kappaB activation.

--- TEST sample sentences ---
  The GATA-1 nuclear bodies differed from previously described nuclear structures and there was [NEG]no [NEG

## Load the SFU Review Corpus 

In [7]:
# Load the SFU Review Corpus for augmentation
_SFU_URL = (
    "https://www.sfu.ca/~mtaboada/docs/research/"
    "SFU_Review_Corpus_Negation_Speculation.zip"
)
_SFU_DIR = "/content/sfu_corpus"

sfu_files = []
try:
    resp = requests.get(_SFU_URL, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        z.extractall(_SFU_DIR)
    sfu_files = (glob.glob(f"{_SFU_DIR}/**/*.txt", recursive=True) +
                 glob.glob(f"{_SFU_DIR}/**/*.xml",  recursive=True))
    print(f"Downloaded {len(sfu_files)} SFU files.")
except requests.RequestException as e:
    print(f"WARNING: could not download SFU corpus ({e}). "
          f"Continuing without SFU augmentation.")

if sfu_files:
    with open(sfu_files[0], encoding="utf-8", errors="replace") as f:
        print(f"\n--- Raw sample: {sfu_files[0]} ---")
        print(f.read()[:800])

sfu_examples = []
n_untagged_files = 0
for fpath in sfu_files:
    with open(fpath, encoding="utf-8", errors="replace") as f:
        file_raw = f.read()

    if _has_sentence_tags(file_raw):
        try:
            sfu_examples.extend(_parse_bioscope_stream(file_raw))
        except ET.ParseError:

            try:
                sfu_examples.extend(
                    _parse_bioscope_stream(f"<root>{file_raw}</root>")
                )
            except ET.ParseError as e:
                print(f"  SKIP (unparseable XML): {fpath} ({e})")
    else:
        n_untagged_files += 1
        for chunk in _split_into_pseudo_sentences(file_raw):
            try:
                sfu_examples.extend(
                    _parse_bioscope_stream(f"<sentence>{chunk}</sentence>")
                )
            except ET.ParseError:
                continue

if sfu_files:
    print(f"({n_untagged_files}/{len(sfu_files)} SFU files had no sentence "
          f"tags and were split heuristically instead.)")

n_with_neg  = sum(1 for ex in sfu_examples if any(ex["negation_scope"]))
n_with_spec = sum(1 for ex in sfu_examples if any(ex["speculation_scope"]))
print(f"\nParsed {len(sfu_examples)} SFU sentences "
      f"({n_with_neg} with negation, {n_with_spec} with speculation).")

if sfu_files:
    print(f"Sentences-per-file ratio: {len(sfu_examples) / len(sfu_files):.2f} "
          f"(should be well above 1.0 for a multi-sentence review corpus)")

print("\n--- Sample SFU sentences (verify before merging) ---")
shown = 0
for ex in sfu_examples:
    if any(ex["negation_scope"]) and shown < 5:
        print(" ", _preview_example(ex))
        shown += 1

if n_with_neg > 0 or n_with_spec > 0:
    before = len(train_ex)
    train_ex = train_ex + sfu_examples
    print(f"\nMerged SFU: {before} -> {len(train_ex)} training sentences.")
else:
    print("\nWARNING: 0 cue/scope spans found — tag format likely differs, "
          "or the SFU download was skipped above. NOT merging.")

Downloaded 401 SFU files.

--- Raw sample: /content/sfu_corpus/SFU_Review_Corpus_Negation_Speculation/README.txt ---
The SFU Review Corpus, annotated for speculation and negation

Natalia Konstantinova and Sheila C.M. de Sousa


This corpus contains the 400 reviews in the SFU Review Corpus (Taboada, 2008),
annotated for speculation and negation. Please see the enclosed guidelines for
information on the annotation, and Konstantinova et al. (2012) for more detailed
information on the corpus and its applications.


License
-------
Natalia Konstantinova and Sheila C.M. de Sousa
Copyright (c) 2012

This corpus is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program i
(1/401 SFU files had no sentence tags and were split heuristically instead.)

Parsed 17281 SFU sentences (5858 with negation, 852 with speculati

## Augment Training Data with Weak Modal Cues

In [8]:
# Augment training data with weak modal cues
WEAK_MODALS       = {"can", "could", "would"}
STRONG_HEDGE_CUES = {"may", "might"}

random.seed(seed)
augmented = []

for ex in train_ex:
    toks_lower = [w.lower() for w in ex["tokens"]]
    cue_positions = [
        i for i, (w, s) in enumerate(zip(toks_lower, ex["speculation_scope"]))
        if w in STRONG_HEDGE_CUES and s == 1
    ]
    if not cue_positions:
        continue
    if any(w in WEAK_MODALS for w in toks_lower):
        continue

    for replacement in WEAK_MODALS:
        new_ex = copy.deepcopy(ex)
        for pos in cue_positions:
            new_ex["tokens"][pos] = replacement
        augmented.append(new_ex)

before = len(train_ex)
train_ex = train_ex + augmented
random.shuffle(train_ex)
print(f"Cue substitution: +{len(augmented)} synthetic sentences "
      f"({before} -> {len(train_ex)} training sentences).")

Cue substitution: +1203 synthetic sentences (25592 -> 26795 training sentences).


## Tokenize and Create Datasets

In [9]:
# Define negation and speculation cue vocabularies
NEGATION_CUES = {
    "not", "no", "without", "fail", "failure", "lack", "lacking",
    "neither", "nor", "never", "none", "cannot", "unable", "absence", "non",
}
SPECULATION_CUES = {
    "may", "might", "could", "can", "would", "suggest", "suggests", "suggested",
    "possible", "possibly", "presumed", "presumably", "propose", "proposed",
    "likely", "probable", "probably", "indicate", "indicated", "appear",
    "appears", "seem", "seems", "whether", "hypothesis", "hypothesize",
    "assume", "assumed", "perhaps",
}

tokenizer = AutoTokenizer.from_pretrained(backbone)

def _tokenise_split(split_examples):
    rows = []
    for ex in split_examples:
        input_ids  = [tokenizer.cls_token_id]
        neg_labels = [IGNORE_INDEX]
        spec_labels = [IGNORE_INDEX]

        for word, n_lab, s_lab in zip(
            ex["tokens"], ex["negation_scope"], ex["speculation_scope"]
        ):
            subwords = tokenizer.tokenize(word)
            if not subwords:
                continue
            ids = tokenizer.convert_tokens_to_ids(subwords)
            input_ids.extend(ids)
            # first subword carries the real label; continuations are ignored
            neg_labels.extend( [n_lab] + [IGNORE_INDEX] * (len(ids) - 1))
            spec_labels.extend([s_lab] + [IGNORE_INDEX] * (len(ids) - 1))

        # truncate to max_len (leave room for [SEP])
        input_ids   = input_ids[:max_len - 1]   + [tokenizer.sep_token_id]
        neg_labels  = neg_labels[:max_len - 1]  + [IGNORE_INDEX]
        spec_labels = spec_labels[:max_len - 1] + [IGNORE_INDEX]

        pad_len       = max_len - len(input_ids)
        attention_mask = [1] * len(input_ids) + [0] * pad_len
        input_ids    += [tokenizer.pad_token_id] * pad_len
        neg_labels   += [IGNORE_INDEX] * pad_len
        spec_labels  += [IGNORE_INDEX] * pad_len

        rows.append((input_ids, attention_mask, neg_labels, spec_labels))
    return rows


class NegSpecDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        ids, mask, neg, spec = self.rows[idx]
        return (
            torch.tensor(ids,  dtype=torch.long),
            torch.tensor(mask, dtype=torch.long),
            torch.tensor(neg,  dtype=torch.long),
            torch.tensor(spec, dtype=torch.long),
        )


train_rows = _tokenise_split(train_ex)
val_rows   = _tokenise_split(val_ex)
test_rows  = _tokenise_split(test_ex)

train_ds = NegSpecDataset(train_rows)
val_ds   = NegSpecDataset(val_rows)
test_ds  = NegSpecDataset(test_rows)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

print(f"Tokenised: train={len(train_rows)}  val={len(val_rows)}  test={len(test_rows)}")

# Preview one aligned row
sample_ids, sample_mask, sample_neg, sample_spec = train_rows[0]
sample_tokens = tokenizer.convert_ids_to_tokens(sample_ids)
print("\n--- Sample tokenised row (train[0]) ---")
print(f"{'token':<15} {'neg':>4} {'spec':>5} {'mask':>5}")
for t, n, s, m in zip(sample_tokens, sample_neg, sample_spec, sample_mask):
    if m == 0:
        break
    print(f"{t:<15} {n:>4} {s:>5} {m:>5}")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

Tokenised: train=26795  val=1780  test=1780

--- Sample tokenised row (train[0]) ---
token            neg  spec  mask
[CLS]           -100  -100     1
in                 0     0     1
addition           0     0     1
,               -100  -100     1
rf                 0     0     1
##x1            -100  -100     1
can                0     0     1
bind               0     0     1
simultaneously     0     0     1
,               -100  -100     1
most               0     0     1
likely             0     1     1
as                 0     1     1
a                  0     1     1
heterodimer        0     1     1
,                  0     0     1
with               0     0     1
the                0     0     1
transcription      0     0     1
factor             0     0     1
mib                0     0     1
##p1            -100  -100     1
to                 0     0     1
nr                 0     0     1
##e             -100  -100     1
gamma              0     0     1
.               -100  -1

## Define the Multitask Token Classification Model

In [11]:
# Define the multitask token classification model
class MultiHeadTokenClassifier(nn.Module):
    """Shared PubMedBERT backbone with separate token-classification heads
    for negation scope and speculation scope.

    """

    def __init__(self, backbone_name, num_labels=2, dropout=0.1):
        super().__init__()
        self.backbone   = AutoModel.from_pretrained(backbone_name)
        hidden          = self.backbone.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.head_neg   = nn.Linear(hidden, num_labels)
        self.head_spec  = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        x = self.backbone(input_ids=input_ids,
                          attention_mask=attention_mask).last_hidden_state
        x = self.dropout(x)
        return self.head_neg(x), self.head_spec(x)


model = MultiHeadTokenClassifier(backbone, dropout=dropout).to(device)

if freeze_layers > 0:
    for p in model.backbone.embeddings.parameters():
        p.requires_grad = False
    for layer in model.backbone.encoder.layer[:freeze_layers]:
        for p in layer.parameters():
            p.requires_grad = False

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in model.parameters())
print(f"Params: {n_trainable:,} trainable / {n_total:,} total "
      f"(froze bottom {freeze_layers} encoder layers)")

no_decay = {"bias", "LayerNorm.weight"}
trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
param_groups = [
    {"params": [p for n, p in trainable if not any(nd in n for nd in no_decay)],
     "weight_decay": weight_decay},
    {"params": [p for n, p in trainable if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]

optimizer = AdamW(param_groups, lr=lr)

loss_fn_neg  = CrossEntropyLoss(
    ignore_index=IGNORE_INDEX,
    weight=neg_class_weights.to(device),
    label_smoothing=label_smoothing,
)
loss_fn_spec = CrossEntropyLoss(
    ignore_index=IGNORE_INDEX,
    weight=spec_class_weights.to(device),
    label_smoothing=label_smoothing,
)

total_steps  = len(train_loader) * epochs
warmup_steps = int(total_steps * warmup_ratio)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Scheduler: {warmup_steps} warmup / {total_steps} total steps")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Params: 43,120,900 trainable / 109,485,316 total (froze bottom 6 encoder layers)
Scheduler: 5025 warmup / 50250 total steps


## Train the Model

In [14]:
# Train the model
best_val_loss = float("inf")
patience_left = patience

for epoch in range(epochs):

    model.train()
    train_loss_total = 0.0

    for ids, mask, neg, spec in train_loader:
        ids, mask, neg, spec = (t.to(device) for t in (ids, mask, neg, spec))

        logits_neg, logits_spec = model(ids, mask)
        loss = (loss_fn_neg(logits_neg.view(-1, 2),  neg.view(-1)) +
                loss_fn_spec(logits_spec.view(-1, 2), spec.view(-1)))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        train_loss_total += loss.item()

    model.eval()
    val_loss_total = 0.0
    vl_neg_p, vl_neg_l, vl_spec_p, vl_spec_l = [], [], [], []

    with torch.no_grad():
        for ids, mask, neg, spec in val_loader:
            ids, mask, neg, spec = (t.to(device) for t in (ids, mask, neg, spec))

            logits_neg, logits_spec = model(ids, mask)
            val_loss_total += (loss_fn_neg(logits_neg.view(-1, 2),  neg.view(-1)) +
                               loss_fn_spec(logits_spec.view(-1, 2), spec.view(-1))).item()

            for logits, golds, preds_list, golds_list in [
                (logits_neg,  neg,  vl_neg_p,  vl_neg_l),
                (logits_spec, spec, vl_spec_p, vl_spec_l),
            ]:
                p = logits.argmax(dim=-1).view(-1)
                g = golds.view(-1)
                m = g != IGNORE_INDEX
                preds_list += p[m].cpu().tolist()
                golds_list += g[m].cpu().tolist()

    val_neg_f1  = f1_score(vl_neg_l,  vl_neg_p,  pos_label=1, zero_division=0)
    val_spec_f1 = f1_score(vl_spec_l, vl_spec_p, pos_label=1, zero_division=0)
    avg_val_loss = val_loss_total / len(val_loader)

    print(f"Epoch {epoch+1:02d}  lr={scheduler.get_last_lr()[0]:.2e}  "
          f"train_loss={train_loss_total/len(train_loader):.4f}  "
          f"val_loss={avg_val_loss:.4f}  "
          f"val_neg_f1={val_neg_f1:.4f}  val_spec_f1={val_spec_f1:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_left = patience
        torch.save(model.state_dict(), out_path)
        print(f"  -> new best (val_loss={best_val_loss:.4f}), saved to {out_path}")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print("Early stopping.")
            break


Epoch 01  lr=1.97e-05  train_loss=0.3518  val_loss=0.2421  val_neg_f1=0.8715  val_spec_f1=0.9007
  -> new best (val_loss=0.2421), saved to best_model.pt
Epoch 02  lr=1.83e-05  train_loss=0.2757  val_loss=0.2275  val_neg_f1=0.8992  val_spec_f1=0.8989
  -> new best (val_loss=0.2275), saved to best_model.pt
Epoch 03  lr=1.68e-05  train_loss=0.2159  val_loss=0.2603  val_neg_f1=0.8916  val_spec_f1=0.9166
Epoch 04  lr=1.53e-05  train_loss=0.1768  val_loss=0.3019  val_neg_f1=0.8904  val_spec_f1=0.9153
Epoch 05  lr=1.38e-05  train_loss=0.1368  val_loss=0.3134  val_neg_f1=0.9166  val_spec_f1=0.9083
Early stopping.


## Evaluate on the Test Set

In [15]:
# Evaluate on the test set
print("\nLoading best checkpoint for test evaluation...")
model.load_state_dict(torch.load(out_path, map_location=device))
model.eval()

test_loss_total = 0.0
ts_neg_p, ts_neg_l, ts_spec_p, ts_spec_l = [], [], [], []

with torch.no_grad():
    for ids, mask, neg, spec in test_loader:
        ids, mask, neg, spec = (t.to(device) for t in (ids, mask, neg, spec))

        logits_neg, logits_spec = model(ids, mask)
        test_loss_total += (loss_fn_neg(logits_neg.view(-1, 2),  neg.view(-1)) +
                            loss_fn_spec(logits_spec.view(-1, 2), spec.view(-1))).item()

        for logits, golds, preds_list, golds_list in [
            (logits_neg,  neg,  ts_neg_p,  ts_neg_l),
            (logits_spec, spec, ts_spec_p, ts_spec_l),
        ]:
            p = logits.argmax(dim=-1).view(-1)
            g = golds.view(-1)
            m = g != IGNORE_INDEX
            preds_list += p[m].cpu().tolist()
            golds_list += g[m].cpu().tolist()

test_neg_f1  = f1_score(ts_neg_l,  ts_neg_p,  pos_label=1, zero_division=0)
test_spec_f1 = f1_score(ts_spec_l, ts_spec_p, pos_label=1, zero_division=0)
print(f"TEST  loss={test_loss_total/len(test_loader):.4f}  "
      f"neg_f1={test_neg_f1:.4f}  spec_f1={test_spec_f1:.4f}")

scored = []
with torch.no_grad():
    for i in range(len(test_ds)):
        ids, mask, neg, spec = test_ds[i]
        if not (neg == 1).any() and not (spec == 1).any():
            continue

        logits_neg, logits_spec = model(
            ids.unsqueeze(0).to(device), mask.unsqueeze(0).to(device)
        )
        pred_neg  = logits_neg.argmax(dim=-1).squeeze(0).cpu()
        pred_spec = logits_spec.argmax(dim=-1).squeeze(0).cpu()

        valid      = neg != IGNORE_INDEX
        mismatches = ((pred_neg[valid] != neg[valid]) |
                      (pred_spec[valid] != spec[valid])).sum().item()
        scored.append((mismatches, i, ids, neg, pred_neg, spec, pred_spec))

scored.sort(key=lambda r: -r[0])

print(f"\n--- Sample predictions ({len(scored)} test sentences with a cue) ---")
for mismatches, i, ids, neg, pred_neg, spec, pred_spec in scored[:5]:
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"\nSentence {i} — {mismatches} mismatch(es):")
    print(f"  {'token':<15} {'g_neg':>5} {'p_neg':>5} {'g_spec':>6} {'p_spec':>6}")
    for t, gn, pn, gs, ps in zip(
        tokens, neg.tolist(), pred_neg.tolist(), spec.tolist(), pred_spec.tolist()
    ):
        if gn == IGNORE_INDEX:
            continue
        flag = "  <--" if (gn != pn or gs != ps) else ""
        print(f"  {t:<15} {gn:>5} {pn:>5} {gs:>6} {ps:>6}{flag}")


Loading best checkpoint for test evaluation...
TEST  loss=0.2782  neg_f1=0.8794  spec_f1=0.9156

--- Sample predictions (520 test sentences with a cue) ---

Sentence 1194 — 36 mismatch(es):
  token           g_neg p_neg g_spec p_spec
  these               0     0      0      0
  data                0     0      0      0
  provide             0     0      0      0
  the                 0     0      0      0
  molecular           0     0      0      0
  basis               0     0      0      0
  for                 0     0      0      0
  the                 0     0      0      0
  notion              0     0      0      1  <--
  that                0     0      0      1  <--
  (                   0     0      0      1  <--
  part                0     0      0      1  <--
  of                  0     0      0      1  <--
  the                 0     0      0      1  <--
  anti                0     0      0      1  <--
  effect              0     0      0      1  <--
  of                 

## Inspect Per-Cue Recall Performance

In [16]:
# Inspect sample predictions
def _clean_token(tok):
    return tok.replace("##", "").lower()


neg_cue_stats  = {}
spec_cue_stats = {}

model.eval()
with torch.no_grad():
    for i in range(len(test_ds)):
        ids, mask, neg, spec = test_ds[i]
        logits_neg, logits_spec = model(
            ids.unsqueeze(0).to(device), mask.unsqueeze(0).to(device)
        )
        pred_neg  = logits_neg.argmax(dim=-1).squeeze(0).cpu()
        pred_spec = logits_spec.argmax(dim=-1).squeeze(0).cpu()
        tokens    = tokenizer.convert_ids_to_tokens(ids)

        for tok, gn, pn, gs, ps in zip(
            tokens, neg.tolist(), pred_neg.tolist(), spec.tolist(), pred_spec.tolist()
        ):
            if gn == IGNORE_INDEX:
                continue
            word = _clean_token(tok)
            if word in NEGATION_CUES and gn == 1:
                s = neg_cue_stats.setdefault(word, [0, 0])
                s[0] += 1
                s[1] += pn == 1
            if word in SPECULATION_CUES and gs == 1:
                s = spec_cue_stats.setdefault(word, [0, 0])
                s[0] += 1
                s[1] += ps == 1

for title, stats in [("Negation", neg_cue_stats), ("Speculation", spec_cue_stats)]:
    print(f"\n--- {title}: per-cue recall (worst first) ---")
    print(f"{'cue':<15} {'occ':>5} {'correct':>8} {'recall':>8}")
    for cue, (occ, correct) in sorted(stats.items(), key=lambda kv: kv[1][1] / kv[1][0]):
        print(f"{cue:<15} {occ:>5} {correct:>8} {correct/occ:>8.1%}")



--- Negation: per-cue recall (worst first) ---
cue               occ  correct   recall
failure             4        0     0.0%
fail                2        1    50.0%
lacking             5        4    80.0%
no                 26       26   100.0%
absence            11       11   100.0%
without            14       14   100.0%
not               163      163   100.0%
lack               12       12   100.0%
non                 2        2   100.0%
neither             2        2   100.0%
nor                 2        2   100.0%
none                1        1   100.0%
unable              4        4   100.0%
never               1        1   100.0%

--- Speculation: per-cue recall (worst first) ---
cue               occ  correct   recall
could              12       10    83.3%
can                16       14    87.5%
whether            21       20    95.2%
suggests           16       16   100.0%
likely             12       12   100.0%
may                79       79   100.0%
suggest            58

## Analyze Cue-Level Recall with Training Data

In [17]:
# Analyze cue-level recall
train_neg_counts  = {c: 0 for c in NEGATION_CUES}
train_spec_counts = {c: 0 for c in SPECULATION_CUES}

for ex in train_ex:
    for word, n_lab, s_lab in zip(
        ex["tokens"], ex["negation_scope"], ex["speculation_scope"]
    ):
        w = word.lower().strip(".,;:()")
        if n_lab == 1 and w in NEGATION_CUES:
            train_neg_counts[w] += 1
        if s_lab == 1 and w in SPECULATION_CUES:
            train_spec_counts[w] += 1

for title, train_counts, test_stats in [
    ("Negation",    train_neg_counts,  neg_cue_stats),
    ("Speculation", train_spec_counts, spec_cue_stats),
]:
    print(f"\n--- {title}: train occurrences vs test recall ---")
    print(f"{'cue':<15} {'train_occ':>10} {'test_occ':>9} {'recall':>8}")
    for cue in sorted(train_counts, key=train_counts.get):
        t_occ          = train_counts[cue]
        te_occ, correct = test_stats.get(cue, [0, 0])
        recall_str     = f"{correct/te_occ:.1%}" if te_occ else "n/a"
        print(f"{cue:<15} {t_occ:>10} {te_occ:>9} {recall_str:>8}")

print("\nRule of thumb: <15–20 train occurrences + low recall → scarcity.")
print("Many train occurrences + low recall → genuine modelling gap.")


--- Negation: train occurrences vs test recall ---
cue              train_occ  test_occ   recall
non                      1         2   100.0%
failure                  3         4     0.0%
none                     6         1   100.0%
fail                     8         2    50.0%
cannot                  14         0      n/a
unable                  18         4   100.0%
lacking                 27         5    80.0%
neither                 32         2   100.0%
nor                     39         2   100.0%
lack                    41        12   100.0%
absence                 42        11   100.0%
never                   42         1   100.0%
without                 87        14   100.0%
no                     198        26   100.0%
not                   1029       163   100.0%

--- Speculation: train occurrences vs test recall ---
cue              train_occ  test_occ   recall
assume                   0         0      n/a
probable                 1         0      n/a
presumed           

## Test on an Example Article from Europe PMC

In [18]:
# Test on an example article from Europe PMC
pmcid = "PMC11975295"

resp = requests.get(
    f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML"
)
if resp.status_code != 200 or not resp.content:
    print(f"Could not retrieve {pmcid} (status {resp.status_code}).")
    paragraphs = []
else:
    root       = ET.fromstring(resp.content)
    paragraphs = [
        "".join(p.itertext()).strip()
        for p in root.iter("p")
        if len("".join(p.itertext()).strip()) > 30
    ]

print(f"Fetched {len(paragraphs)} paragraphs from {pmcid}.")

full_text = re.sub(r"\s+", " ", " ".join(paragraphs))
sentences = [
    s.strip()
    for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", full_text)
    if len(s.strip()) > 15
]
print(f"Split into {len(sentences)} sentences.\n")

model.eval()
with torch.no_grad():
    for sent in sentences:
        words          = sent.split()
        input_ids      = [tokenizer.cls_token_id]
        word_of_subword = [None]

        for w_idx, word in enumerate(words):
            subwords = tokenizer.tokenize(word)
            if not subwords:
                continue
            ids = tokenizer.convert_tokens_to_ids(subwords)
            input_ids.extend(ids)
            word_of_subword.extend([w_idx] * len(ids))

        input_ids       = input_ids[:max_len - 1]       + [tokenizer.sep_token_id]
        word_of_subword = word_of_subword[:max_len - 1] + [None]
        attention_mask  = [1] * len(input_ids)

        ids_t  = torch.tensor([input_ids],     dtype=torch.long).to(device)
        mask_t = torch.tensor([attention_mask], dtype=torch.long).to(device)

        logits_neg, logits_spec = model(ids_t, mask_t)
        pred_neg  = logits_neg.argmax(dim=-1).squeeze(0).cpu().tolist()
        pred_spec = logits_spec.argmax(dim=-1).squeeze(0).cpu().tolist()

        # collapse subword predictions to word level
        word_neg  = [0] * len(words)
        word_spec = [0] * len(words)
        for sub_idx, w_idx in enumerate(word_of_subword):
            if w_idx is None:
                continue
            if pred_neg[sub_idx]  == 1:
                word_neg[w_idx]  = 1
            if pred_spec[sub_idx] == 1:
                word_spec[w_idx] = 1

        marked = []
        for word, n, s in zip(words, word_neg, word_spec):
            if n and s:   marked.append(f"[NEG+SPEC]{word}")
            elif n:       marked.append(f"[NEG]{word}")
            elif s:       marked.append(f"[SPEC]{word}")
            else:         marked.append(word)

        weak = sorted({w.lower() for w in words} & {"could", "can", "would"})
        flag = f"  <-- {weak}" if weak else ""
        print(f"{' '.join(marked)}{flag}")

Fetched 76 paragraphs from PMC11975295.
Split into 201 sentences.

Prajwal Pradeep Gundale; Email: dr.prajwalgundale@gmail.com.
Disclosure: The authors report no [NEG]conflicts [NEG]of [NEG]interest.
Editor-in-Chief: Sonia M.
Associate Editor: Elisa de Paula Resende.
Alzheimer's disease (AD) is a progressive neurodegenerative disorder characterized by cognitive decline, memory loss, and behavioral changes.
Despite being the most common cause of dementia, effective treatments have been elusive.
However, recent years have witnessed significant advancements in understanding and treating Alzheimer's.
Key to these advancements is the shift toward targeted therapies tailored to individual genetic and biomarker profiles, promising more effective outcomes while minimizing side effects.
The integration of advanced neuroimaging techniques has revolutionized early diagnosis and disease monitoring, enabling proactive intervention strategies that [SPEC]may [SPEC]alter [SPEC]disease [SPEC]trajectori